## Imports

In [ ]:
import pandas as pd
from google.colab import drive
import os

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Extract Data

In [ ]:
DATA_RAW_PATH = '/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/raw/'

# Full diagnosis codes (ICD codes and titles)
df_diag_all = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_diagnoses_all_true.csv'))

# Core admission info (textual data, identifiers)
df_admissions = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_diagnoses.csv'),
                            usecols=['hadm_id', 'subject_id', 'chief_complaint', 'physical_exam', 'HPI'])

# Procedures performed
df_procedures = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_procedures.csv'))

# Laboratory events table (first occurrence only)
df_lab = pd.read_csv(os.path.join(DATA_RAW_PATH, 'heart_labevents_first_lab.csv'))

In [ ]:
df_diag_all.head()

,subject_id,hadm_id,seq_num,icd_code,long_title
0,10000980,26913865,1,I214,Non-ST elevation (NSTEMI) myocardial infarction
1,10000980,26913865,2,I5023,Acute on chronic systolic (congestive) heart f...
2,10000980,26913865,3,I2542,Coronary artery dissection
3,10000980,26913865,4,N184,"Chronic kidney disease, stage 4 (severe)"
4,10000980,26913865,5,I348,Other nonrheumatic mitral valve disorders


## Transform

### Main Diagnosis Processing

In [ ]:
# Filter the main diagnosis (seq_num = 1) and extract the code and name
df_diag_principal = df_diag_all[df_diag_all['seq_num'] == 1][['hadm_id', 'icd_code', 'long_title']]
df_diag_principal.rename(columns={'icd_code': 'icd_diag_principal', 'long_title': 'name_diag_principal'}, inplace=True)
df_diag_principal.head()

,hadm_id,icd_diag_principal,name_diag_principal
0,26913865,I214,Non-ST elevation (NSTEMI) myocardial infarction
15,29654838,I5033,Acute on chronic diastolic (congestive) heart ...
26,24760295,I214,Non-ST elevation (NSTEMI) myocardial infarction
41,23822395,I2109,ST elevation (STEMI) myocardial infarction inv...
61,28723315,I359,"Nonrheumatic aortic valve disorder, unspecified"


### Processing of Procedures

In [ ]:
# Groups procedures into a single line by admission (hadm_id)
df_procs_grouped = df_procedures.groupby('hadm_id')['long_title'].apply(lambda x: ', '.join(x.astype(str).unique())).reset_index()
df_procs_grouped.rename(columns={'long_title': 'list_procedures'}, inplace=True)
df_procs_grouped.head()

,hadm_id,list_procedures
0,20007905,"Pericardiocentesis, Pulmonary artery wedge mon..."
1,20014999,"Dilation of Coronary Artery, One Artery with D..."
2,20015927,Combined right and left heart cardiac catheter...
3,20021932,Percutaneous transluminal coronary angioplasty...
4,20022560,Percutaneous transluminal coronary angioplasty...


### Union Tables

In [ ]:
# 1. Join Admission with the Main Diagnosis
df_base = pd.merge(df_admissions, df_diag_principal, on='hadm_id', how='left')

# 2. Join the result with the Procedures
df_final_base = pd.merge(df_base, df_procs_grouped, on='hadm_id', how='left')
df_final_base

,subject_id,hadm_id,HPI,physical_exam,chief_complaint,icd_diag_principal,name_diag_principal,list_procedures
0,10000980,29654838,":\n___ yo woman with h/o hypertension, hyperli...","Admission exam:\nGENERAL- Oriented x3. Mood, a...",\nShortness of breath\n \n,I5033,Acute on chronic diastolic (congestive) heart ...,NaN
1,10000980,26913865,":\nThis is a ___ M with history of diabetes, d...",ADMISSION EXAM:\nGeneral- appears comfortable...,\ndyspnea\n \n,I214,Non-ST elevation (NSTEMI) myocardial infarction,Percutaneous transluminal coronary angioplasty...
2,10002013,24760295,":\n___ w/ PMH of CAD s/p PCI x3, s/p off-pump ...",Admission:\nVS- T 99.4 BP 157/88 HR 118 RR 24 ...,\nchest pain\n \n,I214,Non-ST elevation (NSTEMI) myocardial infarction,"Left heart cardiac catheterization, Coronary a..."
3,10002155,23822395,:\n___ is a ___ yo female with a past medical ...,"GENERAL: WDWN in NAD. Oriented x3. Mood, affec...",\nchest pressure\n \n,I2109,ST elevation (STEMI) myocardial infarction inv...,Percutaneous transluminal coronary angioplasty...
4,10004457,28723315,:\nMr. ___ is a ___ with a hx of CAD (s/p DES ...,On Admission:\nVS- 97.8 157/64 101 18 98% RA \...,"\nAbnormal Stress Test, New AI\n \n",I359,"Nonrheumatic aortic valve disorder, unspecified",NaN
...,...,...,...,...,...,...,...,...
4756,19993951,29858732,:\nMr. ___ is a ___ yo male with PMH significa...,"ON ADMISSION:\nT 97.7F, BP: 91/63, HR: 80, RR ...","\nDyspnea, Fatigue \n \n",I5023,Acute on chronic systolic (congestive) heart f...,NaN
4757,19994505,22534556,":\n___ yo M PMHx DM, HTN, hypercholesterolemia...",ADMISSION EXAM:\nVS: 97.5 138/74 89 20 95%RA ...,\nTransfered for EP evaluation \n \n,I442,"Atrioventricular block, complete",Implantation or replacement of automatic cardi...
4758,19997293,28847872,:\n___ yo M w/ h/o paraplegia ___ T9 epidural ...,"ADMISSION:\nGENERAL: overweight, intubated. Pa...",\nVFib\n \n,I442,"Atrioventricular block, complete",Initial insertion of transvenous lead [electro...
4759,19997367,24169669,:\n___ is a ___ year old woman with multiple m...,ADMISSION PHYSICAL EXAM \n====================...,\nchest pain\n \n,I5033,Acute on chronic diastolic (congestive) heart ...,Combined right and left heart cardiac catheter...


In [ ]:
print("\n--- Base DataFrame (Demographics + Diagnosis + Procedures) Created ---")
print(f"Number of unique hospital admissions: {df_final_base['hadm_id'].nunique()}")


--- Base DataFrame (Demographics + Diagnosis + Procedures) Created ---
Number of unique hospital admissions: 4761


### Attaching Lab Events (Pivoting)

In [ ]:
df_lab[df_lab['hadm_id'] ==  20007905].sort_values('label')

,hadm_id,specimen_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,comments,label,fluid,category,examination_group
27,20007905.0,58793664,2189-07-29 14:59:00,2189-07-29 16:30:00,80,80.00,IU/L,0.00,40.00,abnormal,NaN,Alanine Aminotransferase (ALT),Blood,Chemistry,Liver Function Tests
28,20007905.0,58793664,2189-07-29 14:59:00,2189-07-29 16:30:00,3.3,3.30,g/dL,3.50,5.20,abnormal,NaN,Albumin,Blood,Chemistry,Liver Function Tests
29,20007905.0,17175029,2189-07-30 14:10:00,2189-07-30 16:46:00,2.3,2.30,g/dL,NaN,NaN,NaN,NaN,"Albumin, Body Fluid",Other Body Fluid,Chemistry,"Albumin, Body Fluid"
30,20007905.0,10364076,2189-07-31 11:08:00,2189-07-31 13:38:00,1.9,1.90,g/dL,NaN,NaN,NaN,NaN,"Albumin, Pleural",Pleural,Chemistry,"Albumin, Pleural"
31,20007905.0,58793664,2189-07-29 14:59:00,2189-07-29 16:30:00,154,154.00,IU/L,40.00,130.00,abnormal,NaN,Alkaline Phosphatase,Blood,Chemistry,Liver Function Tests
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,20007905.0,52272262,2189-07-29 15:42:00,2189-07-29 15:49:00,47,47.00,mm Hg,35.00,45.00,abnormal,NaN,pCO2,Blood,Blood Gas,Blood Gas
119,20007905.0,52272262,2189-07-29 15:42:00,2189-07-29 15:49:00,7.38,7.38,units,7.35,7.45,NaN,NaN,pH,Blood,Blood Gas,Blood Gas
120,20007905.0,72093533,2189-07-31 12:42:00,2189-07-31 13:01:00,___,7.50,units,NaN,NaN,NaN,___,pH,Other Body Fluid,Blood Gas,Blood Gas
121,20007905.0,72678444,2189-07-29 23:15:00,2189-07-30 03:47:00,6.0,6.00,units,5.00,8.00,NaN,NaN,pH,Urine,Hematology,Urine Test


In [ ]:
df_lab[df_lab['label'] ==  'C-Reactive Protein'].sort_values('comments')

,hadm_id,specimen_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,comments,label,fluid,category,examination_group
51135,22324256.0,59493066,2150-03-04 13:15:00,2150-03-04 18:09:00,NaN,NaN,mg/L,0.0,5.0,NaN,">300. LOW RISK <1.0, AVERAGE RISK 1.0-3.0, HI...",C-Reactive Protein,Blood,Chemistry,Infection and Immunology
72740,23212325.0,90406696,2144-04-22 21:15:00,2144-04-23 04:38:00,NaN,NaN,mg/L,0.0,5.0,NaN,">300. LOW RISK <1.0, AVERAGE RISK 1.0-3.0, HI...",C-Reactive Protein,Blood,Chemistry,Infection and Immunology
175651,27610308.0,89284491,2136-06-29 19:55:00,2136-06-29 22:43:00,NaN,NaN,mg/L,0.0,5.0,NaN,">300. LOW RISK <1.0, AVERAGE RISK 1.0-3.0, HI...",C-Reactive Protein,Blood,Chemistry,Infection and Immunology
16766,20712362.0,70815912,2120-02-07 04:37:00,2120-02-07 11:09:00,NaN,NaN,mg/L,0.0,5.0,NaN,GREATER THAN 300.,C-Reactive Protein,Blood,Chemistry,Infection and Immunology
140998,26245725.0,12285852,2185-11-29 08:05:00,2185-11-29 11:22:00,NaN,NaN,mg/L,0.0,5.0,NaN,"GREATER THAN 300. LOW RISK <1.0, AVERAGE RISK...",C-Reactive Protein,Blood,Chemistry,Infection and Immunology
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212267,29209434.0,88860145,2182-03-09 08:10:00,2182-03-09 10:42:00,8.6,8.6,mg/L,0.0,5.0,abnormal,NaN,C-Reactive Protein,Blood,Chemistry,Infection and Immunology
212848,29226245.0,45538671,2166-03-17 06:35:00,2166-03-17 09:57:00,113.7,113.7,mg/L,0.0,5.0,abnormal,NaN,C-Reactive Protein,Blood,Chemistry,Infection and Immunology
218007,29470952.0,23505623,2181-07-19 07:42:00,2181-07-19 08:46:00,9.0,9.0,mg/L,0.0,5.0,abnormal,NaN,C-Reactive Protein,Blood,Chemistry,Infection and Immunology
226519,29857283.0,79952208,2153-11-03 11:44:00,2153-11-03 20:47:00,19.5,19.5,mg/L,0.0,5.0,abnormal,NaN,C-Reactive Protein,Blood,Chemistry,Infection and Immunology


In [ ]:
df_lab[(~df_lab['comments'].isna()) & (df_lab['valuenum'].isna())].sort_values('comments')

,hadm_id,specimen_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,comments,label,fluid,category,examination_group
162990,27074582.0,82296751,2113-04-07 11:30:00,2113-04-07 13:11:00,NaN,NaN,NaN,NaN,NaN,abnormal,1+*.,Polychromasia,Blood,Hematology,Blood Cell Morphology
42130,21905656.0,84445181,2132-10-04 05:51:00,2132-10-04 09:13:00,NaN,NaN,NaN,NaN,NaN,abnormal,1+*.,Poikilocytosis,Blood,Hematology,Blood Cell Morphology
75670,23333218.0,17971199,2193-04-08 09:18:00,2193-04-08 14:45:00,NaN,NaN,NaN,NaN,NaN,abnormal,1+*.,Polychromasia,Blood,Hematology,Blood Cell Morphology
41153,21842610.0,37735291,2174-06-03 06:00:00,2174-06-03 11:13:00,NaN,NaN,NaN,NaN,NaN,abnormal,1+*.,Poikilocytosis,Blood,Hematology,Blood Cell Morphology
41139,21842610.0,37735291,2174-06-03 06:00:00,2174-06-03 11:13:00,NaN,NaN,NaN,NaN,NaN,abnormal,1+*.,Macrocytes,Blood,Hematology,Blood Cell Morphology
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120021,25296555.0,72837473,2161-03-04 17:41:00,2161-03-04 18:41:00,NaN,NaN,NaN,NaN,NaN,NaN,c/w monosodium urate crystals.,"Joint Crystals, Comment",Joint Fluid,Hematology,"Joint Crystals, Comment"
51401,22337488.0,69666242,2141-06-29 18:00:00,2141-06-29 19:56:00,NaN,NaN,NaN,NaN,NaN,NaN,c/w monosodium urate crystals.,"Joint Crystals, Comment",Joint Fluid,Hematology,"Joint Crystals, Comment"
31617,21385662.0,15644999,2141-10-10 15:30:00,2141-10-10 16:53:00,NaN,NaN,NaN,NaN,NaN,NaN,c/w monosodium urate crystals.,"Joint Crystals, Comment",Joint Fluid,Hematology,"Joint Crystals, Comment"
124270,25511808.0,89961547,2147-04-05 15:29:00,2147-04-05 16:32:00,NaN,NaN,NaN,NaN,NaN,NaN,c/w monosodium urate crystals.,"Joint Crystals, Comment",Joint Fluid,Hematology,"Joint Crystals, Comment"


In [ ]:
df_lab.value_counts('flag')

,count
flag,
abnormal,61790


In [ ]:
# Apply the text parsing for CRP/critical values (from previous step)
def handle_c_reactive_protein(row):
    if pd.notna(row['valuenum']): return row['valuenum']
    comment = str(row['comments']).upper()
    if 'GREATER THAN 300' or '>300.' in comment: return 301.0
    return row['valuenum']

In [ ]:
# Select relevant columns and filter to keep only rows where the fluid is 'Blood' (systemic eligibility criteria are almost always based on Blood measurements)
df_lab_clean = df_lab.loc[df_lab['fluid'] == 'Blood', ['hadm_id', 'label', 'valuenum', 'flag', 'comments']].copy()
df_lab_clean['hadm_id'] = df_lab_clean['hadm_id'].astype(int)

# Use and clean up the original 'flag' for robustness
df_lab_clean['flag'] = df_lab_clean['flag'].fillna('normal')

CRP_LABEL = 'C-Reactive Protein'
df_lab_clean['valuenum'] = df_lab.apply(lambda row:
                                          handle_c_reactive_protein(row)
                                          if row['label'] == CRP_LABEL else row['valuenum'],
                                          axis=1)

# PCR > 300 should be always abnormal
df_lab_clean.loc[(df_lab_clean['label'] == CRP_LABEL) & (df_lab_clean['valuenum'] == 301.0), 'flag'] = 'abnormal'

# Pivot 1: make each unique 'label' (lab test) a new column
# The 'valuenum' becomes the value in the new column
df_lab_pivot_value = df_lab_clean.pivot(index='hadm_id', columns='label', values='valuenum')
df_lab_pivot_value.reset_index(inplace=True)
df_lab_pivot_value.head()

# PIVOT 2: Status Flag (e.g., Creatinine_FLAG: ABNORMAL)
df_lab_pivot_flag = df_lab_clean.pivot_table(
    index='hadm_id',
    columns='label',
    values='flag',
    aggfunc='first' # Get the first recorded status text
)
df_lab_pivot_flag.reset_index(inplace=True)

# Rename columns from PIVOT 2 to avoid collision
df_lab_pivot_flag.columns = [col + '_FLAG' if col != 'hadm_id' else col for col in df_lab_pivot_flag.columns]



In [ ]:
df_lab_clean.value_counts('flag')

,count
flag,
normal,128761
abnormal,57478


### Final Merge

In [ ]:
# Merge Base DF (Demographics) with both Pivot Value and Pivot Flag DFs
df_final_complete = pd.merge(df_final_base, df_lab_pivot_value, on='hadm_id', how='left')
df_final_complete = pd.merge(df_final_complete, df_lab_pivot_flag, on='hadm_id', how='left')
df_final_complete

,subject_id,hadm_id,HPI,physical_exam,chief_complaint,icd_diag_principal,name_diag_principal,list_procedures,% Hemoglobin A1c,25-OH Vitamin D,...,Von Willebrand Factor Activity_FLAG,Von Willebrand Factor Antigen_FLAG,WBC Count_FLAG,White Blood Cells_FLAG,Young Cells_FLAG,eAG_FLAG,pCO2_FLAG,pH_FLAG,pO2_FLAG,tacroFK_FLAG
0,10000980,29654838,":\n___ yo woman with h/o hypertension, hyperli...","Admission exam:\nGENERAL- Oriented x3. Mood, a...",\nShortness of breath\n \n,I5033,Acute on chronic diastolic (congestive) heart ...,NaN,8.1,NaN,...,NaN,NaN,NaN,normal,NaN,abnormal,NaN,NaN,NaN,NaN
1,10000980,26913865,":\nThis is a ___ M with history of diabetes, d...",ADMISSION EXAM:\nGeneral- appears comfortable...,\ndyspnea\n \n,I214,Non-ST elevation (NSTEMI) myocardial infarction,Percutaneous transluminal coronary angioplasty...,6.8,NaN,...,NaN,NaN,NaN,normal,NaN,abnormal,NaN,NaN,NaN,NaN
2,10002013,24760295,":\n___ w/ PMH of CAD s/p PCI x3, s/p off-pump ...",Admission:\nVS- T 99.4 BP 157/88 HR 118 RR 24 ...,\nchest pain\n \n,I214,Non-ST elevation (NSTEMI) myocardial infarction,"Left heart cardiac catheterization, Coronary a...",NaN,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,NaN,NaN,NaN,NaN
3,10002155,23822395,:\n___ is a ___ yo female with a past medical ...,"GENERAL: WDWN in NAD. Oriented x3. Mood, affec...",\nchest pressure\n \n,I2109,ST elevation (STEMI) myocardial infarction inv...,Percutaneous transluminal coronary angioplasty...,NaN,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,abnormal,normal,abnormal,NaN
4,10004457,28723315,:\nMr. ___ is a ___ with a hx of CAD (s/p DES ...,On Admission:\nVS- 97.8 157/64 101 18 98% RA \...,"\nAbnormal Stress Test, New AI\n \n",I359,"Nonrheumatic aortic valve disorder, unspecified",NaN,NaN,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4756,19993951,29858732,:\nMr. ___ is a ___ yo male with PMH significa...,"ON ADMISSION:\nT 97.7F, BP: 91/63, HR: 80, RR ...","\nDyspnea, Fatigue \n \n",I5023,Acute on chronic systolic (congestive) heart f...,NaN,NaN,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,NaN,NaN,NaN,NaN
4757,19994505,22534556,":\n___ yo M PMHx DM, HTN, hypercholesterolemia...",ADMISSION EXAM:\nVS: 97.5 138/74 89 20 95%RA ...,\nTransfered for EP evaluation \n \n,I442,"Atrioventricular block, complete",Implantation or replacement of automatic cardi...,6.7,NaN,...,NaN,NaN,NaN,normal,NaN,abnormal,NaN,NaN,NaN,NaN
4758,19997293,28847872,:\n___ yo M w/ h/o paraplegia ___ T9 epidural ...,"ADMISSION:\nGENERAL: overweight, intubated. Pa...",\nVFib\n \n,I442,"Atrioventricular block, complete",Initial insertion of transvenous lead [electro...,NaN,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,abnormal,abnormal,abnormal,NaN
4759,19997367,24169669,:\n___ is a ___ year old woman with multiple m...,ADMISSION PHYSICAL EXAM \n====================...,\nchest pain\n \n,I5033,Acute on chronic diastolic (congestive) heart ...,Combined right and left heart cardiac catheter...,NaN,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,normal,abnormal,abnormal,NaN


In [ ]:
print("\n--- df_final_complete Created (Dual Pivot) ---")
print(f"Total number of columns: {len(df_final_complete.columns)}")


--- df_final_complete Created (Dual Pivot) ---
Total number of columns: 680


## Load

In [ ]:
DATA_PROCESSED_PATH = '/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/'
OUTPUT_FILENAME = 'df_diag_final_elegibility.parquet'
OUTPUT_PATH = os.path.join(DATA_PROCESSED_PATH, OUTPUT_FILENAME)

try:
    df_final_complete.to_parquet(OUTPUT_PATH, index=False)
    print(f"\n--- SUCESSO: DataFrame final salvo em: {OUTPUT_PATH} ---")
except Exception as e:
    print(f"\nERRO ao salvar o arquivo: {e}")


--- SUCESSO: DataFrame final salvo em: /content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/df_diag_final_elegibility.parquet ---
